## RGB to Grayscale SOL

**Objective:** Convert a color image from its RGB (Red, Green, Blue) representation to a grayscale image. This process simplifies the image by reducing it to one channel of intensity, facilitating further image processing and analysis tasks that do not require color information.

"SOL" stands for "Speed Of Light," a performance metric that indicates the theoretical maximum throughput or efficiency achievable by a hardware component, such as a GPU, under ideal conditions. It is used as a benchmark to evaluate how close actual performance comes to the hardware's optimal capabilities. In image processing and GPU computing, comparing measured performance to the SOL helps identify bottlenecks and assess how efficiently an algorithm utilizes the available resources.

## Implementation

In [51]:
!nvidia-smi

Sat Oct 24 18:29:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             27W /   70W |     187MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [52]:
!pip install ninja

In [53]:
# importing libraries
import torch
from torch.utils.cpp_extension import load_inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import __version__
import math

print("Library Versions:")
print("Pytorch: ", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name: ", torch.cuda.get_device_name(0))
print("Numpy: ", np.__version__)
print("Pandas: ", pd.__version__)
print("Matplotlib: ", __version__)





Library Versions:
Pytorch:  2.11.0+cu130
CUDA Available:  True
GPU Name:  Tesla T4
Numpy:  2.1.3
Pandas:  2.2.3
Matplotlib:  3.10.0


NOTES:

CODE: `torch::Tensor grayscale = torch::zeros({1, static_cast<int64_t>(width), static_cast<int64_t>(height)}, img.options());`

REASONING: `width` and `height` are 32-bit `int` values, while `torch::zeros` takes the output shape as an `IntArrayRef`, a list of `int64_t`. A brace list of plain `int`s does not convert to that type, so the build fails. `static_cast<int64_t>` widens each dimension to the 64-bit size type PyTorch stores on a tensor. The kernel can keep `int` parameters; the cast is only required at this shape argument.


In [54]:
cpp_code = """
#include <torch/extension.h>

torch::Tensor rgb_to_grayscale_cuda(torch::Tensor img);
"""

cuda_code = """
#include <torch/extension.h>
#include <cstdint>
#include <c10/cuda/CUDAException.h>

// This is the CUDA Kernel code for the RGB to Grayscale conversion.
__global__ void rgb_to_grayscale_cuda_kernel(uint8_t* rgb, uint8_t* grayscale, int width, int height){
    // Step 1: Get the thread id from the block and grid
    int thread_id = blockIdx.x * blockDim.x + threadIdx.x;
    // Step 2: Check if the thread id is within the bounds of the input tensor
    if (thread_id < width * height){
        // Step 3: convert to grayscale with the particular formula
        float r = rgb[thread_id];
        float g = rgb[thread_id + width * height];
        float b = rgb[thread_id + 2 * width * height];
        float gray = 0.2989f * r + 0.5870f * g + 0.1140f * b;
        grayscale[thread_id] = static_cast<uint8_t>(gray);
    }
}

// RGB to Grayscale Launcher
torch::Tensor rgb_to_grayscale_cuda(torch::Tensor img){
    // Step 1: Implement Checks before launching kernels
    // 1.1 Check if input and output are on the same GPU
    TORCH_CHECK(img.device().is_cuda(), "Input must be on a GPU");
    // 1.2 Check if input is contiguous
    TORCH_CHECK(img.is_contiguous(), "Input must be contiguous");
    // 1.3 Check if input has uint8 data type
    TORCH_CHECK(img.dtype() == torch::kU8, "Input must have uint8 data type");
    // 1.4 Check if input has 3 channels
    TORCH_CHECK(img.size(0) == 3, "Input must have only 3 channels");
    // 1.5 Check if input has only 3 dimensions --channels, width, height
    TORCH_CHECK(img.dim() == 3, "Input must have 3 dimensions");
    // Step 2: Get the dimensions of the input tensor
    int width = img.size(1);
    int height = img.size(2);
    // Step 3: Create output tensor
    // Create output tensor with the same shape as the input tensor (averaged across the channels)
    torch::Tensor grayscale = torch::empty(
        {1, static_cast<int64_t>(width), static_cast<int64_t>(height)},
        img.options()
    );
    // Step 4: Launch the kernel
    // 4.1 Configure grid and block dimensions
    int block_size = 256;
    int grid_size = (width * height + block_size - 1) / block_size;
    // 4.2 Launch the kernel
    rgb_to_grayscale_cuda_kernel<<<grid_size, block_size>>>(img.data_ptr<uint8_t>(), grayscale.data_ptr<uint8_t>(), width, height);
    // 4.3 Check for errors
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    // 4.4 Return the output tensor
    return grayscale;
}
"""

In [55]:
ext_name = "grayscale_module_v2"

# Load the C++ and CUDA code
grayscale_module = load_inline(
    name=ext_name,
    cpp_sources=cpp_code,
    cuda_sources=cuda_code,
    functions=["rgb_to_grayscale_cuda"],
    extra_cuda_cflags=["-O3"],
    verbose=True
)



In [56]:
img=torch.randint(0, 255, (3, 224, 224), dtype=torch.uint8, device="cuda")
grayscale_img=grayscale_module.rgb_to_grayscale_cuda(img)







In [57]:
grayscale_img.shape

torch.Size([1, 224, 224])

Getting bandwidth of GPU using torch cuda properties.

`props=torch.cuda.get_device_properties(0)`

The variable 'props' contains the GPU device properties obtained from torch.cuda.get_device_properties(0).
Some key attributes of 'props' relevant to memory bandwidth are:
- props.memory_bus_width: The width of the memory data bus in bits. This determines how many bits can be transferred in one memory cycle.
- props.memory_clock_rate: The memory clock rate in kHz; this is how fast the memory runs.

Memory bandwidth (in bytes per second) is calculated based on these properties using the formula:
Bandwidth = 2 * memory_clock_rate * 1000 * (memory_bus_width / 8)

Here's what each term means:
- memory_clock_rate is multiplied by 1000 to convert kHz to Hz (cycles per second).
- memory_bus_width / 8 converts the bus width from bits to bytes.
- The factor of 2 accounts for double data rate (DDR) memory, which transfers data on both rising and falling clock edges.

This gives the theoretical peak memory bandwidth of the GPU in bytes per second, which we often convert to GB/s for readability.

In [58]:
props=torch.cuda.get_device_properties(0)
print(props)
print(f"GPU Name: {props.name}")
print(f"Total Memory: {props.total_memory / 1024**2} MB")


_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=6e3cbd6c-90aa-c022-f122-92a0a82d639e, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB)
GPU Name: Tesla T4
Total Memory: 14912.6875 MB


In [59]:
props.memory_bus_width

256

In [60]:
def calculate_memory_bandwidth(props):
    beta_peak = 2 * props.memory_clock_rate * 1e3 * (props.memory_bus_width / 8)
    return beta_peak / 1e9

print(f"{calculate_memory_bandwidth(props):.0f} GB/s")  # 320 on a T4



320 GB/s


In [61]:
def calculate_estimated_time(launch, transfer, props):
    """
    Calculate estimated total time as launch time plus transfer time.

    Args:
        launch (float): Launch time (micro seconds)
        transfer (float): Transfer size (bytes)
        props: CUDA device properties object (should have memory_bus_width and memory_clock_rate)

    Returns:
        float: Estimated total time (seconds)
    """
    # Calculate theoretical peak memory bandwidth in bytes per second
    bandwidth = 2 * props.memory_clock_rate * 1e3 * (props.memory_bus_width / 8)  # bytes per second

    transfer_time = transfer / bandwidth *1e6 # in seconds

    estimated_time = launch + transfer_time
    return estimated_time



In [62]:
estimated_time=calculate_estimated_time(17.5, 224*224*4, props)
print(f"{estimated_time:.2f} microseconds")

18.13 microseconds


In [63]:
H = W = 4096
img = torch.randint(0, 255, (3, H, W), dtype=torch.uint8, device="cuda")

for _ in range(10):
    grayscale_module.rgb_to_grayscale_cuda(img)
torch.cuda.synchronize()

start, end = torch.cuda.Event(True), torch.cuda.Event(True)
iters = 50
start.record()
for _ in range(iters):
    grayscale_module.rgb_to_grayscale_cuda(img)
end.record()
torch.cuda.synchronize()

measured_us = start.elapsed_time(end) * 1e3 / iters   # ms -> µs per call
B = 4 * H * W                                         # bytes
bandwidth = 2 * props.memory_clock_rate * 1e3 * (props.memory_bus_width / 8)
memory_us = B / bandwidth * 1e6
estimated_us = 17.5 + memory_us

print(f"measured   {measured_us:.1f} µs")
print(f"estimated  {estimated_us:.1f} µs")
print(f"ratio      {measured_us / estimated_us:.2f}")

measured   564.9 µs
estimated  227.2 µs
ratio      2.49


## Observations

- On a 4096×4096 image the measured call is **564.9 µs** and the speed-of-light estimate is **227.2 µs**, a ratio of **2.49**. The kernel is in the memory-bound regime the lecture describes: a small factor from the floor, rather than the 12× gap from the FP64 plus `zeros` run or the ~200× gap on the 224×224 image.
- The estimate is 17.5 µs of launch plus 209.7 µs to move `4 × 4096 × 4096` bytes at 320 GB/s. Launch is only about 8% of that estimate, so this image is large enough that the result is about memory traffic rather than kernel-launch overhead.
- 564.9 µs for those bytes is about **119 GB/s**, roughly 37% of the T4's 320 GB/s peak. The ratio of 2.49 is that shortfall: the hardware ceiling assumes every byte moves at the spec-sheet rate for the whole call.
- The remaining gap is expected. Peak bandwidth is a ceiling, the three channel planes are read with a large stride, and each timed iteration still allocates a new output with `torch::empty` before the kernel is queued. The CUDA-event interval includes the time the GPU waits for that allocation.
- Arithmetic intensity stays **1.25 FLOPs/byte** (5 FLOPs, 4 bytes per pixel). That is far below the T4 FP32 ridge near 25 FLOPs/byte, which is why the right floor is `T_launch + B/β` and not a compute-time bound.